# Chapter 14 Hands-On Coding Exam — Reproducible Python Engineering (100 Points)

**Book Reference:** Chapter 14, Section 14.7 (Hands-On Coding Exam)  
**Suggested Time:** 35–45 minutes (Open Book, Closed Solution)

| Task | Engineering Workflow Function | Points |
| :--- | :--- | :---: |
| **Task 14.1** | `audit_dependencies(req_path)` (Dependency lock & reproducibility audit) | 30 |
| **Task 14.2** | `scaffold_research_package(base_dir, pkg_name)` (`src/` package generator) | 35 |
| **Task 14.3** | `run_triage_cli(argv)` (Typed `argparse` clinical triage runner) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q sharukh-tutor
import sharukh_tutor as tutor

tutor.activate(chapter="ch14", notebook="coding_exam")


---
## Exam Task 14.1 (30 Points): Audit Dependency Reproducibility

Write `audit_dependencies(req_path)` that parses a `requirements.txt` file (ignoring blank lines and `#` comments, lowercasing package names, splitting on `==`, `>=`, `<=`, `~=` or bare names) and **returns** a dictionary:
- `"total"`: `int` count of unique dependencies.
- `"pinned_count"`: `int` count of dependencies whose specifier starts with `"=="`.
- `"unpinned"`: **sorted list** of lowercased package names not pinned with `"=="`.
- `"reproducible"`: `bool` (`True` when `len(unpinned) == 0`, else `False`).

In [ ]:
def audit_dependencies(req_path):
    """Audit requirements.txt and return reproducibility report."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    reqs = {}
    with open(req_path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            for op in ("==", ">=", "<=", "~="):
                if op in line:
                    name, ver = line.split(op, 1)
                    reqs[name.strip().lower()] = op + ver.strip()
                    break
            else:
                reqs[line.lower()] = ""

    unpinned = sorted(
        pkg for pkg, spec in reqs.items()
        if not spec.startswith("==")
    )
    total = len(reqs)
    return {
        "total": total,
        "pinned_count": total - len(unpinned),
        "unpinned": unpinned,
        "reproducible": len(unpinned) == 0,
    }
    ### END SOLUTION


---
## Exam Task 14.2 (35 Points): Scaffold a Research Package (`src/` Layout)

Write `scaffold_research_package(base_dir, pkg_name)` that generates a clean starter repository inside `base_dir`:
1. Normalize `pkg_name` (strip, lowercase, replace `-` and spaces with `_`, collapse repeated `_`, strip leading/trailing `_`).
2. Create these 4 files inside `base_dir`, each containing non-empty UTF-8 starter text:
   - `pyproject.toml`
   - `README.md`
   - `src/<pkg>/__init__.py`
   - `tests/test_smoke.py`
3. **Return** the **sorted list** of created relative **POSIX** paths (using forward slashes `/`).

In [ ]:
from pathlib import Path
import re

def scaffold_research_package(base_dir, pkg_name):
    """Create a src-layout research package and return POSIX paths."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    cleaned = pkg_name.strip().lower()
    cleaned = cleaned.replace("-", "_").replace(" ", "_")
    pkg = re.sub(r"_+", "_", cleaned).strip("_")

    root = Path(base_dir)
    files = {
        "pyproject.toml": (
            f'[project]\nname = "{pkg}"\nversion = "0.1.0"\n'
        ),
        "README.md": f"# {pkg}\n\nReproducible research package.\n",
        f"src/{pkg}/__init__.py": (
            f'"""Core module for {pkg}."""\n__version__ = "0.1.0"\n'
        ),
        "tests/test_smoke.py": (
            f"import {pkg}\n\n"
            f"def test_version():\n"
            f"    assert hasattr({pkg}, '__version__')\n"
        ),
    }
    created = []
    for rel_path, content in files.items():
        target = root / rel_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        created.append(target.relative_to(root).as_posix())
    return sorted(created)
    ### END SOLUTION


---
## Exam Task 14.3 (35 Points): Clinical Vitals Triage CLI Runner

Write `run_triage_cli(argv)` that parses CLI argument list `argv` via `argparse.ArgumentParser`:
1. Register required flags: `--temp` (`float`), `--hb` (`float`), and `--sbp` (`int`), and parse `argv` with `parser.parse_args(argv)`.
2. Compute `priority`:
   - `"Urgent"` if `temp >= 39.5` or `sbp >= 170` or `hb < 8.5`
   - `"Moderate"` if `temp >= 38.0` or `sbp >= 140` or `hb < 11.0`
   - `"Routine"` otherwise
3. **Return** `{"temp_c": args.temp, "hb_g_dl": args.hb, "sbp_mmhg": args.sbp, "priority": priority}`.

In [ ]:
import argparse

def run_triage_cli(argv):
    """Parse clinical vitals from argv and return triage dict."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    parser = argparse.ArgumentParser(
        description="Clinical vitals triage CLI"
    )
    parser.add_argument("--temp", type=float, required=True)
    parser.add_argument("--hb", type=float, required=True)
    parser.add_argument("--sbp", type=int, required=True)
    args = parser.parse_args(argv)

    if args.temp >= 39.5 or args.sbp >= 170 or args.hb < 8.5:
        priority = "Urgent"
    elif args.temp >= 38.0 or args.sbp >= 140 or args.hb < 11.0:
        priority = "Moderate"
    else:
        priority = "Routine"

    return {
        "temp_c": args.temp,
        "hb_g_dl": args.hb,
        "sbp_mmhg": args.sbp,
        "priority": priority,
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch14_exam_01": audit_dependencies,
    "ch14_exam_02": scaffold_research_package,
    "ch14_exam_03": run_triage_cli,
}

tutor.check_exam("ch14", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 14.1 (Case Sensitivity & Reproducibility Flag):** Mixing `NumPy>=1.26` and `numpy>=1.26` without `.lower()` breaks alphabetical sorting and duplicate detection. Set `reproducible = len(unpinned) == 0` so any loose specifier (`>=`, `<=`, `~=`, or bare name) immediately marks the lockfile non-reproducible.
- **Task 14.2 (Missing Parent Directories & Windows Backslashes):** Writing `src/<pkg>/__init__.py` without `target.parent.mkdir(parents=True, exist_ok=True)` raises `FileNotFoundError`. Convert returned paths with `target.relative_to(root).as_posix()` so Windows and Linux return identical forward-slash strings.
- **Task 14.3 (Calling `parse_args()` Without `argv` & Boundary Order):** Always pass `argv` into `parser.parse_args(argv)`—omitting `argv` reads Jupyter's own process flags and crashes. Check the `Urgent` branch *before* `Moderate` so a patient with `temp == 39.5` is never downgraded to `Moderate`.